## obtendo_locais.ipynb

Utilizamos uma base de dados da ANAC que nos dá a latitude e longitude de cara aeroporto brasileiro. Então, a partir disso, utilizamos a tabela `origem_destinos_unicos.csv` para filtrar o local dos aeródromos, pois só nos interessam aqueles utilizados na aviação comercial. Esses aeródromos ficam salvos na tabela `locais_aerodromos.csv`, e será utilizado em `view_mapa_brasil.ipynb` para visualizar as rotas.

In [28]:
import pandas as pd
import lxml
import numpy as np

Crio um dataframe com a latitude e a longitude de todos os aerodromos do Brasil, obtidos da ANAC

In [29]:
url = "https://www.anac.gov.br/assuntos/setor-regulado/aerodromos/localizacao-geografica-arquivos/aerodromos/publicos-todas-as-regioes/download.table"

# Lê a tabela da ANAC, que eles não disponibilizam para download diretamente ebaaa
df = pd.read_html(url)[0]

# Seleciona as colunas desejadas
df = df[["CÓDIGO OACI", "NOME", "coordenada"]]

# Separa latitude e longitude
df[["Latitude", "Longitude"]] = (
    df["coordenada"]
    .str.split(",", expand=True)
)

# Renomeia as colunas
df = df.rename(columns={
    "CÓDIGO OACI": "ICAO",
    "NOME": "Nome"
})

# Remove a coluna original de coordenadas
df = df.drop(columns=["coordenada"])

# Converte coordenadas para números
df["Latitude"] = pd.to_numeric(df["Latitude"])
df["Longitude"] = pd.to_numeric(df["Longitude"])

df_aerodromos_com_local = df.copy()

In [30]:
df_aerodromos_com_local.sample(5)

,ICAO,Nome,Latitude,Longitude
471,SSPB,JUVENAL LOUREIRO CARDOSO,-26.217778,-52.694167
211,SIZX,JUARA SUL,-11.286667,-57.538889
248,SNCS,CAMPOS SALES,-7.049444,-40.364167
408,SSBE,CAMAPUÃ,-19.597778,-54.019167
53,SBIH,ITAITUBA,-4.242222,-56.000833


In [31]:
df = pd.read_csv("../dados_processados/origens_destinos_unicos.csv", sep=";")

Aplico o filtro para obter apenas os locais de aeródromos que estão na nossa base de dados.

In [32]:
# Precisa utilizar numpy aqui!

origens_unicas = pd.unique(df["origem"])
destinos_unicos = pd.unique(df["destino"])

aerodromos_unicos = np.union1d(origens_unicas, destinos_unicos)
print(aerodromos_unicos.shape)



(148,)


Salvo isso na tabela `locais_aerodromos.csv`.

In [36]:
aerodromos_com_local_considerados = df_aerodromos_com_local[df_aerodromos_com_local["ICAO"].isin(aerodromos_unicos)]

aerodromos_com_local_considerados.sample(5)
aerodromos_com_local_considerados.shape

aerodromos_com_local_considerados.to_csv("../dados_processados/locais_aerodromos.csv", sep=";")